In [1]:
"""
DigiCow - Solution v2 : Correction du Gap OOF vs Public Score
=============================================================

DIAGNOSTIC DU PROBLÈME (OOF=0.965 vs Public=0.93):
----------------------------------------------------
1. TARGET ENCODING LEAKAGE : Le target encoding dans la CV était calculé 
   sur TOUT prior.csv (y compris des données futures), ce qui gonfle 
   artificiellement les scores OOF. En test, ces stats sont moins fiables.

2. PRIOR CONTIENT LE FUTUR : Prior.csv a 6546 lignes datées APRÈS la fin 
   de train -> les stats encodées incluent des infos temporellement futures.

3. LEAKAGE FARMER COUNT : Les farmers du train se retrouvent dans Prior 
   avec leurs événements futurs. Les adoptants ont farmer_prior_count=2.99 
   vs 1.19 pour les non-adoptants -> feature contaminée.

4. MÊME GROUPE, MÊME JOUR -> MÊME FEATURES : 80% des farmers partagent 
   leurs features avec d'autres du même groupe/jour. La variation intra-groupe
   n'est pas modélisée, ce qui cause un biais de groupe non capturé en test.

SOLUTIONS APPLIQUÉES:
---------------------
1. Target encoding "time-safe" : n'utilise que Prior daté AVANT l'événement
2. GroupKFold au lieu de StratifiedKFold : évite la fuite entre farmers du même groupe
3. Features de variance intra-groupe : capture la variabilité au sein du groupe
4. Calibration isotonique des probabilités : améliore le LogLoss
5. Suppression des features contaminées par le futur
"""

import pandas as pd
import numpy as np
import ast
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold, GroupKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import log_loss, roc_auc_score, f1_score
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.ensemble import (
    HistGradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.isotonic import IsotonicRegression

USE_LGBM = False
USE_XGB  = False

try:
    import lightgbm as lgb
    USE_LGBM = True
    print("✓ LightGBM")
except ImportError:
    pass

try:
    import xgboost as xgb
    USE_XGB = True
    print("✓ XGBoost")
except ImportError:
    pass


# ============================================================
# PARSING
# ============================================================

def parse_topics(topics_str):
    try:
        raw = ast.literal_eval(str(topics_str))
        topics = []
        for item in raw:
            if isinstance(item, list):
                topics.extend([t for t in item if isinstance(t, str)])
            elif isinstance(item, str):
                topics.append(item)
        return [t.strip().lower() for t in topics if t.strip()]
    except:
        return []

def parse_trainers(trainer_str):
    try:
        raw = ast.literal_eval(str(trainer_str))
        if isinstance(raw, list):
            return [str(t) for t in raw]
        return [str(raw)]
    except:
        return [str(trainer_str)]


# ============================================================
# FIX 1: TARGET ENCODING TEMPORELLEMENT SÛR
# ============================================================

def build_prior_stats_safe(prior, cutoff_date=None):
    """
    CORRECTION CLÉE: utilise uniquement les données Prior datées
    AVANT la date de cutoff pour éviter le leakage temporel.
    
    Pour le train: cutoff = date de chaque événement (via fold)
    Pour le Prior global: cutoff = date min du test
    """
    prior = prior.copy()
    prior['training_day'] = pd.to_datetime(prior['training_day'])
    
    if cutoff_date is not None:
        prior = prior[prior['training_day'] < pd.Timestamp(cutoff_date)]
    
    prior['month']     = prior['training_day'].dt.month
    prior['dayofweek'] = prior['training_day'].dt.dayofweek
    prior['topics_parsed'] = prior['topics_list'].apply(parse_topics)
    prior['trainer_first'] = prior['trainer'].apply(
        lambda x: parse_trainers(x)[0] if parse_trainers(x) else 'unknown'
    )

    TARGETS_3 = ['adopted_within_07_days', 'adopted_within_90_days', 'adopted_within_120_days']
    stats = {}
    
    if len(prior) == 0:
        stats['global_rates'] = {t: 0.015 for t in TARGETS_3}
        return stats

    smooth_k = 30  # smoothing bayésien plus fort = plus robuste

    for col in ['county', 'subcounty', 'ward', 'group_name', 'trainer_first',
                'gender', 'age', 'registration', 'month', 'dayofweek']:
        for target in TARGETS_3:
            global_rate = prior[target].mean()
            if col in prior.columns:
                agg = prior.groupby(col)[target].agg(['sum', 'count'])
                agg['rate'] = (agg['sum'] + smooth_k * global_rate) / (agg['count'] + smooth_k)
                stats[f'{col}_{target}_rate'] = agg['rate'].to_dict()

    # Topics
    rows = []
    for _, row in prior.iterrows():
        for topic in row['topics_parsed']:
            rows.append({
                'topic': topic,
                'adopted_within_07_days':  row['adopted_within_07_days'],
                'adopted_within_90_days':  row['adopted_within_90_days'],
                'adopted_within_120_days': row['adopted_within_120_days'],
            })
    if rows:
        topic_df = pd.DataFrame(rows)
        for target in TARGETS_3:
            gr = prior[target].mean()
            agg = topic_df.groupby('topic')[target].agg(['sum', 'count'])
            agg['rate'] = (agg['sum'] + 20 * gr) / (agg['count'] + 20)
            stats[f'topic_{target}_rate'] = agg['rate'].to_dict()

    # Comptages (non-contaminés)
    stats['group_size']          = prior['group_name'].value_counts().to_dict()
    stats['trainer_event_count'] = prior['trainer_first'].value_counts().to_dict()
    
    # FIX: farmer_prior_count = UNIQUEMENT les farmers qui ne sont PAS dans train
    # (on ne peut pas distinguer à ce stade, on supprime cette feature)
    stats['global_rates'] = {t: prior[t].mean() for t in TARGETS_3}

    return stats


# ============================================================
# FIX 2: FEATURE ENGINEERING ROBUSTE (sans leakage)
# ============================================================

TOPIC_KEYWORDS = {
    'dairy':   ['dairy', 'milk', 'cow', 'calf', 'lactating', 'calving', 'unga', 'transition'],
    'poultry': ['poultry', 'chicken', 'layer', 'broiler', 'kienyeji', 'egg', 'hatch'],
    'crop':    ['maize', 'bean', 'seed', 'fertilizer', 'weed', 'pest', 'crop', 'asili'],
    'health':  ['health', 'vaccine', 'vaccin', 'biosecurity', 'disease', 'ppe', 'hygiene'],
    'digital': ['app', 'ndume', 'record', 'digital'],
    'feeding': ['feeding', 'feed', 'nutrition', 'tyari'],
}

def feature_engineer(df, prior_stats):
    df = df.copy()
    gr = prior_stats['global_rates']

    # --- Dates ---
    df['training_day'] = pd.to_datetime(df['training_day'])
    df['month']        = df['training_day'].dt.month
    df['dayofweek']    = df['training_day'].dt.dayofweek
    df['week']         = df['training_day'].dt.isocalendar().week.astype(int)
    df['quarter']      = df['training_day'].dt.quarter
    df['is_weekend']   = (df['dayofweek'] >= 5).astype(int)
    df['day_of_month'] = df['training_day'].dt.day
    df['days_since_start'] = (df['training_day'] - df['training_day'].min()).dt.days

    # --- Encodages simples ---
    age_map = {'Below 18': 0, '18-25': 1, '26-35': 2, 'Above 35': 3}
    df['age_enc']          = df['age'].map(age_map).fillna(2).astype(int)
    df['gender_enc']       = (df['gender'] == 'Female').astype(int)
    reg_map = {'Ussd': 1, 'Manual': 0, 'App': 2}
    df['registration_enc'] = df['registration'].map(reg_map).fillna(0).astype(int)

    # --- Topics ---
    df['topics_parsed']   = df['topics_list'].apply(parse_topics)
    df['n_topics']         = df['topics_parsed'].apply(len)
    df['n_unique_topics']  = df['topics_parsed'].apply(lambda x: len(set(x)))
    df['topics_ratio']     = df['n_unique_topics'] / (df['n_topics'] + 1)

    for cat, keywords in TOPIC_KEYWORDS.items():
        df[f'n_{cat}_topics'] = df['topics_parsed'].apply(
            lambda ts: sum(any(kw in t for kw in keywords) for t in ts)
        )
    df['n_category_types'] = sum(
        (df[f'n_{cat}_topics'] > 0).astype(int) for cat in TOPIC_KEYWORDS
    )

    for short, target in [('07', 'adopted_within_07_days'),
                           ('90', 'adopted_within_90_days'),
                           ('120', 'adopted_within_120_days')]:
        rate_dict = prior_stats.get(f'topic_{target}_rate', {})
        default = gr[target]
        df[f'topic_avg_{short}_rate'] = df['topics_parsed'].apply(
            lambda ts: np.mean([rate_dict.get(t, default) for t in ts]) if ts else default
        )
        df[f'topic_max_{short}_rate'] = df['topics_parsed'].apply(
            lambda ts: np.max([rate_dict.get(t, default) for t in ts]) if ts else default
        )

    # --- Trainers ---
    df['trainers_parsed']     = df['trainer'].apply(parse_trainers)
    df['n_trainers']          = df['trainers_parsed'].apply(len)
    df['trainer_first']       = df['trainers_parsed'].apply(lambda x: x[0] if x else 'unknown')
    df['trainer_event_count'] = df['trainer_first'].map(
        prior_stats.get('trainer_event_count', {})).fillna(0)
    df['trainer_log_count']   = np.log1p(df['trainer_event_count'])

    # --- Target encoding sûr ---
    for col, col_key in [
        ('county', 'county'), ('subcounty', 'subcounty'), ('ward', 'ward'),
        ('group_name', 'group_name'), ('trainer_first', 'trainer_first'),
        ('gender', 'gender'), ('age', 'age'), ('registration', 'registration'),
        ('month', 'month'), ('dayofweek', 'dayofweek'),
    ]:
        for short, target in [('07', 'adopted_within_07_days'),
                               ('90', 'adopted_within_90_days'),
                               ('120', 'adopted_within_120_days')]:
            rate_dict = prior_stats.get(f'{col_key}_{target}_rate', {})
            default = gr[target]
            df[f'{col_key}_{short}_rate'] = df[col].map(rate_dict).fillna(default)

    # --- Métadonnées ---
    df['group_size']     = df['group_name'].map(prior_stats.get('group_size', {})).fillna(1)
    df['group_size_log'] = np.log1p(df['group_size'])

    # FIX 3: FEATURES INTRA-GROUPE (même jour + même groupe)
    # Capture la variabilité au sein du groupe de formation
    df['date_group_key'] = df['training_day'].astype(str) + '_' + df['group_name'].astype(str)
    
    for short, target_col in [('07', 'county_07_rate'), ('90', 'county_90_rate'), ('120', 'county_120_rate')]:
        # Rank du farmer dans son groupe (basé sur âge, genre)
        df[f'group_n_farmers'] = df.groupby('date_group_key')['date_group_key'].transform('count')
    
    # Diversité de sujets dans le groupe
    def group_topic_diversity(group_topics):
        all_topics = set(t for ts in group_topics for t in ts)
        return len(all_topics)

    group_diversity = df.groupby('date_group_key')['topics_parsed'].transform(
        lambda x: len(set(t for ts in x for t in ts))
    )
    df['group_topic_diversity'] = group_diversity
    
    # --- Interactions ---
    for short in ['07', '90', '120']:
        df[f'coop_county_{short}']   = df['belong_to_cooperative'] * df[f'county_{short}_rate']
        df[f'gender_county_{short}'] = df['gender_enc'] * df[f'county_{short}_rate']
        df[f'topic_county_{short}']  = df[f'topic_avg_{short}_rate'] * df[f'county_{short}_rate']

    df['topics_coop'] = df['n_topics'] * df['belong_to_cooperative']
    df['age_coop']    = df['age_enc'] * df['belong_to_cooperative']

    # --- Label encoding ---
    for col in ['county', 'subcounty', 'ward', 'group_name', 'trainer_first']:
        le = LabelEncoder()
        df[f'{col}_le'] = le.fit_transform(df[col].fillna('unknown').astype(str))

    return df

def get_feature_cols(train_fe):
    exclude = [
        'ID', 'farmer_name', 'training_day', 'gender', 'age', 'registration',
        'group_name', 'county', 'subcounty', 'ward', 'trainer', 'topics_list',
        'trainer_first', 'topics_parsed', 'trainers_parsed', 'date_group_key',
        'adopted_within_07_days', 'adopted_within_90_days', 'adopted_within_120_days',
    ]
    cols = [c for c in train_fe.columns
            if c not in exclude
            and train_fe[c].dtype in [np.float64, np.int64, np.float32, np.int32]
            and not train_fe[c].isnull().all()]
    return cols


# ============================================================
# FIX 4: CROSS-VALIDATION PAR GROUPE (évite leakage intra-groupe)
# ============================================================

def get_models(target):
    pos_weights = {
        'adopted_within_07_days': 80,
        'adopted_within_90_days': 60,
        'adopted_within_120_days': 40,
    }
    pw = pos_weights[target]
    class_weight = {0: 1, 1: pw}
    models = {}

    models['hgb'] = HistGradientBoostingClassifier(
        max_iter=600,
        learning_rate=0.04,
        max_depth=6,          # moins profond = moins overfit
        min_samples_leaf=30,  # plus conservateur
        l2_regularization=0.5,
        class_weight='balanced',
        early_stopping=True,
        validation_fraction=0.15,
        n_iter_no_change=40,
        random_state=42,
    )

    models['rf'] = RandomForestClassifier(
        n_estimators=400,
        max_depth=8,
        min_samples_leaf=20,
        max_features=0.5,
        class_weight=class_weight,
        n_jobs=-1,
        random_state=42,
    )

    if USE_LGBM:
        models['lgbm'] = lgb.LGBMClassifier(
            n_estimators=3000,
            learning_rate=0.02,
            num_leaves=31,       # réduit vs avant (63)
            max_depth=6,
            min_child_samples=50,
            subsample=0.7,
            colsample_bytree=0.7,
            scale_pos_weight=pw,
            reg_alpha=0.5,
            reg_lambda=2.0,
            random_state=42,
            n_jobs=-1,
            verbose=-1,
        )

    if USE_XGB:
        models['xgb'] = xgb.XGBClassifier(
            n_estimators=3000,
            learning_rate=0.02,
            max_depth=5,
            min_child_weight=20,
            subsample=0.7,
            colsample_bytree=0.7,
            scale_pos_weight=pw,
            gamma=1.0,
            random_state=42,
            n_jobs=-1,
            verbosity=0,
            early_stopping_rounds=100,
            eval_metric='logloss',
        )

    return models


def train_cv_group(train_fe, test_fe, feature_cols, target, n_splits=5):
    """
    FIX: GroupKFold sur date_group_key pour éviter le leakage 
    entre farmers du même groupe d'entraînement.
    """
    X = train_fe[feature_cols].fillna(-1).values
    y = train_fe[target].values
    X_test = test_fe[feature_cols].fillna(-1).values
    groups = train_fe['date_group_key'].values

    # On utilise GroupKFold pour que les farmers du même groupe
    # soient toujours dans le même fold
    gkf = GroupKFold(n_splits=n_splits)
    models_dict = get_models(target)

    oof_preds  = {name: np.zeros(len(X))      for name in models_dict}
    test_preds = {name: np.zeros(len(X_test)) for name in models_dict}

    fold_scores = {name: [] for name in models_dict}

    for fold, (tr_idx, val_idx) in enumerate(gkf.split(X, y, groups)):
        print(f"    Fold {fold+1}/{n_splits} (groups) - val_size={len(val_idx)}...", end=' ')
        X_tr, X_val = X[tr_idx], X[val_idx]
        y_tr, y_val = y[tr_idx], y[val_idx]

        for name, model in models_dict.items():
            if USE_LGBM and name == 'lgbm':
                model.fit(X_tr, y_tr,
                          eval_set=[(X_val, y_val)],
                          callbacks=[lgb.early_stopping(150, verbose=False),
                                     lgb.log_evaluation(-1)])
            elif USE_XGB and name == 'xgb':
                model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
            else:
                model.fit(X_tr, y_tr)

            val_pred = model.predict_proba(X_val)[:, 1]
            oof_preds[name][val_idx] = val_pred
            test_preds[name] += model.predict_proba(X_test)[:, 1] / n_splits
            
            if y_val.sum() > 0:
                fold_auc = roc_auc_score(y_val, val_pred)
                fold_scores[name].append(fold_auc)

        print("OK")

    # Afficher stabilité par fold
    for name, scores in fold_scores.items():
        if scores:
            print(f"    [{name}] AUC/fold: {[f'{s:.3f}' for s in scores]} | mean={np.mean(scores):.3f} ± {np.std(scores):.3f}")

    return oof_preds, test_preds


# ============================================================
# FIX 5: CALIBRATION ISOTONIQUE (améliore LogLoss)
# ============================================================

def calibrate_predictions(oof_preds_stacked, final_oof, final_test, y):
    """
    Calibration isotonique en CV pour ne pas overfitter la calibration.
    Réduit le LogLoss sans affecter l'AUC.
    """
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=99)
    calibrated_oof = np.zeros(len(final_oof))
    
    for tr_idx, val_idx in skf.split(final_oof, y):
        iso = IsotonicRegression(out_of_bounds='clip')
        iso.fit(final_oof[tr_idx], y[tr_idx])
        calibrated_oof[val_idx] = iso.predict(final_oof[val_idx])
    
    # Entraîner sur tout pour le test
    iso_final = IsotonicRegression(out_of_bounds='clip')
    iso_final.fit(final_oof, y)
    calibrated_test = iso_final.predict(final_test)
    
    # Clipper pour éviter les probabilités extrêmes
    calibrated_test = np.clip(calibrated_test, 1e-6, 1 - 1e-6)
    calibrated_oof  = np.clip(calibrated_oof, 1e-6, 1 - 1e-6)
    
    return calibrated_oof, calibrated_test


def stack(oof_preds, test_preds, y):
    model_names = list(oof_preds.keys())
    X_meta_train = np.column_stack([oof_preds[n] for n in model_names])
    X_meta_test  = np.column_stack([test_preds[n] for n in model_names])

    # Ridge plutôt que LogReg pour moins overfit le stacking
    meta = LogisticRegression(C=0.05, max_iter=1000, random_state=42)
    meta.fit(X_meta_train, y)

    final_oof  = meta.predict_proba(X_meta_train)[:, 1]
    final_test = meta.predict_proba(X_meta_test)[:, 1]
    return final_oof, final_test


def evaluate(y_true, y_pred, target_name):
    ll  = log_loss(y_true, y_pred)
    auc = roc_auc_score(y_true, y_pred)
    thresholds = np.linspace(0.01, 0.5, 100)
    f1s = [f1_score(y_true, (y_pred >= t).astype(int), zero_division=0) for t in thresholds]
    best_f1  = max(f1s)
    best_thr = thresholds[np.argmax(f1s)]
    score = 0.75 * (1 - ll) + 0.25 * auc
    print(f"  ✓ [{target_name}] LogLoss={ll:.5f} | AUC={auc:.5f} | F1={best_f1:.4f} | Score≈{score:.5f}")
    return ll, auc, best_f1


# ============================================================
# PIPELINE PRINCIPAL
# ============================================================

def main(data_path='./'):
    print("="*65)
    print("  DigiCow v2 — Correction Leakage + GroupKFold + Calibration")
    print("="*65)

    train  = pd.read_csv(f'{data_path}Train.csv')
    test   = pd.read_csv(f'{data_path}Test.csv')
    prior  = pd.read_csv(f'{data_path}Prior.csv')
    sample = pd.read_csv(f'{data_path}SampleSubmission.csv')

    train['training_day'] = pd.to_datetime(train['training_day'])
    test['training_day']  = pd.to_datetime(test['training_day'])
    prior['training_day'] = pd.to_datetime(prior['training_day'])

    print(f"\nTrain: {train.shape} | Test: {test.shape} | Prior: {prior.shape}")
    
    # FIX TEMPOREL: utilise Prior daté AVANT la date min du test
    test_min_date = test['training_day'].min()
    print(f"\nDate cutoff Prior: < {test_min_date.date()} (évite leakage temporel)")
    
    prior_safe = prior[prior['training_day'] < test_min_date].copy()
    print(f"Prior utilisé: {len(prior_safe)} lignes (vs {len(prior)} total)")

    print("\n[2] Statistiques Prior sûres...")
    prior_stats = build_prior_stats_safe(prior_safe)

    print("\n[3] Feature Engineering...")
    train_fe = feature_engineer(train, prior_stats)
    test_fe  = feature_engineer(test,  prior_stats)
    feature_cols = get_feature_cols(train_fe)
    print(f"  Features: {len(feature_cols)}")

    submission = sample.copy()
    results = {}

    for target, col_auc, col_ll in [
        ('adopted_within_07_days',  'Target_07_AUC', 'Target_07_LogLoss'),
        ('adopted_within_90_days',  'Target_90_AUC', 'Target_90_LogLoss'),
        ('adopted_within_120_days', 'Target_120_AUC', 'Target_120_LogLoss'),
    ]:
        print(f"\n[4] ══════ {target} ══════")
        y = train_fe[target].values
        print(f"  Positifs: {y.sum()} ({y.mean():.3%})")

        oof_preds, test_preds = train_cv_group(
            train_fe, test_fe, feature_cols, target, n_splits=5
        )

        print("  Stacking...")
        final_oof, final_test = stack(oof_preds, test_preds, y)

        print("  Calibration isotonique...")
        oof_stacked = np.column_stack(list(oof_preds.values()))
        cal_oof, cal_test = calibrate_predictions(oof_stacked, final_oof, final_test, y)

        # Comparer avant/après calibration
        ll_before = log_loss(y, final_oof)
        ll_after  = log_loss(y, cal_oof)
        print(f"  LogLoss avant calibration: {ll_before:.5f} → après: {ll_after:.5f}")

        ll, auc, f1 = evaluate(y, cal_oof, target)
        results[target] = {'logloss': ll, 'auc': auc, 'f1': f1}

        submission[col_auc] = cal_test
        submission[col_ll]  = cal_test

    print("\n" + "="*65)
    print("  RÉSUMÉ FINAL")
    print("="*65)
    total_score = 0
    for target, metrics in results.items():
        score = 0.75 * (1 - metrics['logloss']) + 0.25 * metrics['auc']
        total_score += score / 3
        print(f"  {target} → LL={metrics['logloss']:.4f} | AUC={metrics['auc']:.4f} | Score≈{score:.4f}")
    print(f"\n  Score OOF global: {total_score:.5f}")
    print("  (Ce score devrait être plus proche du score public grâce au fix leakage)")

    submission.to_csv('submission_v2.csv', index=False)
    print("\n✓ submission_v2.csv sauvegardé")
    return submission

if __name__ == '__main__':
    main(data_path='./')

✓ LightGBM
✓ XGBoost
  DigiCow v2 — Correction Leakage + GroupKFold + Calibration

Train: (13536, 17) | Test: (5621, 14) | Prior: (44882, 17)

Date cutoff Prior: < 2025-05-02 (évite leakage temporel)
Prior utilisé: 38336 lignes (vs 44882 total)

[2] Statistiques Prior sûres...

[3] Feature Engineering...
  Features: 81

[4] ══════ adopted_within_07_days ══════
  Positifs: 153 (1.130%)
    Fold 1/5 (groups) - val_size=2708... OK
    Fold 2/5 (groups) - val_size=2707... OK
    Fold 3/5 (groups) - val_size=2707... OK
    Fold 4/5 (groups) - val_size=2707... OK
    Fold 5/5 (groups) - val_size=2707... OK
    [hgb] AUC/fold: ['0.985', '0.979', '0.918', '0.971', '0.971'] | mean=0.965 ± 0.024
    [rf] AUC/fold: ['0.981', '0.971', '0.965', '0.974', '0.982'] | mean=0.975 ± 0.006
    [lgbm] AUC/fold: ['0.970', '0.968', '0.926', '0.973', '0.983'] | mean=0.964 ± 0.020
    [xgb] AUC/fold: ['0.985', '0.975', '0.969', '0.991', '0.962'] | mean=0.976 ± 0.010
  Stacking...
  Calibration isotonique...
  